In [1]:
# Import Libraries

import asyncio
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# Documents

documents = [
    "Python is a programming language.",
    "Machine learning uses data to learn patterns.",
    "Deep learning uses neural networks.",
    "RAG combines retrieval with generation."
]

In [4]:
# Load Embedding Model

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# Create Embeddings

embeddings = model.encode(documents)

print(embeddings.shape)

(4, 384)


In [7]:
# Create FAISS Index

!pip install faiss-cpu
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Documents indexed:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.2 MB/s eta 0:00:00
Documents indexed: 4


In [8]:
# Query

query = "What is deep learning?"

query_embedding = model.encode(
    [query]
).astype("float32")

In [9]:
# Retrieve Documents

distances, indices = index.search(
    query_embedding,
    3
)

retrieved_documents = [
    documents[i] for i in indices[0]
]

print(retrieved_documents)

['Deep learning uses neural networks.', 'Machine learning uses data to learn patterns.', 'Python is a programming language.']


In [10]:
# Rerank Documents

query_words = set(query.lower().split())

scores = []

for document in retrieved_documents:
    document_words = set(document.lower().split())
    score = len(query_words & document_words)
    scores.append((document, score))

scores.sort(
    key=lambda x: x[1],
    reverse=True
)

print(scores)

[('Deep learning uses neural networks.', 1), ('Python is a programming language.', 1), ('Machine learning uses data to learn patterns.', 0)]


In [11]:
# Select Best Context

context = scores[0][0]

print("Context:", context)

Context: Deep learning uses neural networks.


In [12]:
# Async RAG Function

async def rag(query):
    await asyncio.sleep(0)

    query_embedding = model.encode(
        [query]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        3
    )

    documents_found = [
        documents[i] for i in indices[0]
    ]

    return documents_found

In [13]:
# Run RAG

result = await rag(
    "What is deep learning?"
)

print(result)

['Deep learning uses neural networks.', 'Machine learning uses data to learn patterns.', 'Python is a programming language.']


In [14]:
# Results

print("Query:", query)
print("Retrieved Documents:", retrieved_documents)
print("Best Context:", context)

Query: What is deep learning?
Retrieved Documents: ['Deep learning uses neural networks.', 'Machine learning uses data to learn patterns.', 'Python is a programming language.']
Best Context: Deep learning uses neural networks.
